In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import norm, Mixture
from tqdm import trange

n_trials = 10
experiment_folder = './experiment_temp'

def load_posteriors(experiment_folder, number):
    posterior_df_x1 = pd.read_csv(experiment_folder+f"/posterior/trial_{number}_x1.csv")
    posterior_df_x2 = pd.read_csv(experiment_folder+f"/posterior/trial_{number}_x2.csv")
    posterior_df_joint = pd.read_csv(experiment_folder+f"/posterior/trial_{number}_joint.csv")
    return (posterior_df_x1, posterior_df_x2, posterior_df_joint)

def mixture_pdf(x, mu1, mu2, rho1, rho2):
    p1 = norm.pdf(x, loc=mu1)
    p2 = norm.pdf(x, loc=mu2)
    p = rho1*p1 + rho2*p2
    return p

def single_mixture_estimate(posterior_df, data):
    loglikelihood = posterior_df['log_prob_data'].to_numpy()  
    i_hat = np.argmax(loglikelihood)
    n = np.size(data)
    mu1 = posterior_df["mu.1"].to_numpy()
    mu2 = posterior_df["mu.2"].to_numpy()
    rho1 = posterior_df["rho.1"].to_numpy()
    rho2 = posterior_df["rho.2"].to_numpy()

    w_hat = {
        'mu1':mu1[i_hat], 'mu2':mu2[i_hat],
        'rho1':rho1[i_hat], 'rho2':rho2[i_hat],
    }
    
    Ln_w0 = np.mean([np.log(mixture_pdf(x,**w_hat)) for x in data])
    beta = 1/np.log(n)

    Ln = np.mean(loglikelihood/n)
    rlct = -1*n*beta*(Ln-Ln_w0)
    return rlct

def joint_mixture_estimate(posterior_df, x1, x2):
    loglikelihood = posterior_df['log_prob_data'].to_numpy()
    i_hat = np.argmax(loglikelihood)
    n = np.size(x1)
    mu1 = posterior_df["mu.1"].to_numpy()
    mu2 = posterior_df["mu.2"].to_numpy()
    rho1 = posterior_df["rho.1"].to_numpy()
    rho2 = posterior_df["rho.2"].to_numpy()
    nu1 = posterior_df["nu.1"].to_numpy()
    nu2 = posterior_df["nu.2"].to_numpy()
    gamma1 = posterior_df["gamma.1"].to_numpy()
    gamma2 = posterior_df["gamma.2"].to_numpy()

    w1_hat = {
        'mu1':mu1[i_hat], 'mu2':mu2[i_hat],
        'rho1':rho1[i_hat], 'rho2':rho2[i_hat],
    }
    w2_hat = {
        'mu1':nu1[i_hat], 'mu2':nu2[i_hat],
        'rho1':gamma1[i_hat], 'rho2':gamma2[i_hat],
    } 
    Ln_w0 = np.mean([
        np.log(mixture_pdf(x1[i],**w1_hat))+np.log(mixture_pdf(x2[i], **w2_hat))
        for i in range(n)]
    )
    beta = 1/np.log(n)
    Ln = np.mean(loglikelihood/n)
    rlct = -1*n*beta*(Ln-Ln_w0)
    return rlct

def single_trial_estimates(experiment_folder, number):
    data_df = pd.read_csv(experiment_folder+f"/data/trial_{number}.csv")
    posteriors = load_posteriors(experiment_folder, number)
    rlct_x1 = single_mixture_estimate(posteriors[0], data_df['x1'].to_numpy())
    rlct_x2 = single_mixture_estimate(posteriors[1], data_df['x2'].to_numpy())
    rlct_joint = joint_mixture_estimate(posteriors[2], data_df['x1'].to_numpy(), data_df['x2'].to_numpy())
    return (rlct_x1, rlct_x2, rlct_joint)



In [1]:
x1_lcs = []
x2_lcs = []
joint_lcs = []
for i in trange(n_trials):
    rlct_x1, rlct_x2, rlct_joint = single_trial_estimates(experiment_folder, i)
    x1_lcs.append(rlct_x1)
    x2_lcs.append(rlct_x2)
    joint_lcs.append(rlct_joint)

RLCT = x1_lcs + x2_lcs + joint_lcs
dataset = ['x1 only']*n_trials + ['x2 only']*n_trials + ['Indepedent Joint']*n_trials

RLCT_results = pd.DataFrame({
    'RLCT':RLCT,
    'dataset':dataset
})
print(RLCT_results.head())

NameError: name 'trange' is not defined

In [2]:
fig = px.box(
    RLCT_results,
    title = "Learning Coefficients for Marginal & Product models",
    subtitle="Exact theoretical values indicated by dotted red lines.",
    labels={
        "RLCT": "Learning Coefficient",
        "dataset": "Model"
    },
    x='dataset',
    y='RLCT',
    points="all"
)
fig.add_shape(
    type="line", line_color="salmon", line_width=3, line_dash="dot",
    x0=-.5,x1=1.5,y0=0.75,y1=0.75,yref="y",
)
fig.add_shape(
    type="line", line_color="salmon", line_width=3, line_dash="dot",
    x0=1.5,x1=2.5,y0=1.5,y1=1.5,yref="y",
)
fig.show()

NameError: name 'px' is not defined